# 01 - Data Generation
Generate a synthetic card/transfer transaction dataset for 2025 with realistic settlement-failure patterns, then inspect it. The failure signal is driven by amount, prior failures, night-time traffic, slow authorizations, velocity spikes, new accounts and high-risk corridors - the same drivers the model will learn in notebook 03.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 180_000
print('seed=42, N=', N)

seed=42, N= 180000


In [2]:
# --- timestamps: uniform over 2025, weighted toward daytime hours ---
days = rng.integers(0, 365, N)
hour_w = np.array([2,1,1,1,1,2,3,4,6,8,9,10,10,9,9,9,10,10,9,8,7,6,5,3], dtype=float)
hour_w /= hour_w.sum()
hours = rng.choice(np.arange(24), N, p=hour_w)
timestamps = (pd.to_datetime('2025-01-01')
              + pd.to_timedelta(days, unit='D')
              + pd.to_timedelta(hours, unit='h')
              + pd.to_timedelta(rng.integers(0, 60, N), unit='m'))

# --- categorical fields ---
merchant_cats = ['grocery','fashion','food_delivery','travel','electronics','fuel','health','gaming']
merchant_cat = rng.choice(merchant_cats, N, p=[0.24,0.16,0.15,0.10,0.10,0.09,0.08,0.08])
pay_methods = ['credit_card','debit_card','upi','net_banking','wallet']
payment_method = rng.choice(pay_methods, N, p=[0.34,0.28,0.18,0.11,0.09])
countries = ['US','IN','UK','DE','SG','AE','BR','NG','UA','PK']
country = rng.choice(countries, N, p=[0.26,0.22,0.12,0.10,0.08,0.07,0.06,0.04,0.03,0.02])
is_high_risk = np.isin(country, ['NG','UA','PK'])
currencies = ['USD','INR','EUR','GBP','AED','SGD']
currency = rng.choice(currencies, N, p=[0.34,0.24,0.16,0.10,0.08,0.08])
device_type = rng.choice(['mobile','desktop','tablet','pos'], N, p=[0.55,0.30,0.10,0.05])
issuer_bank = rng.choice(['BankA','BankB','BankC','BankD','BankE','BankF'], N)
merchant_id = 'M' + pd.Series(rng.integers(1, 801, N)).astype(str).str.zfill(4)

# --- numeric behaviour fields ---
amount = rng.lognormal(3.4, 1.05, N)                      # median ~ $30
auth_ms = rng.lognormal(6.7, 0.75, N)                     # median ~ 810 ms
slow = rng.random(N) < 0.03                               # gateway slowdowns
auth_ms = np.where(slow, rng.uniform(3000, 9000, N), auth_ms)
account_age = np.clip(rng.exponential(400, N), 1, 3000).astype(int)
prior_fails = rng.poisson(0.35, N)
repeat = rng.random(N) < 0.03                             # troubled accounts
prior_fails = np.where(repeat, prior_fails + rng.integers(1, 4, N), prior_fails)
velocity = rng.poisson(2.5, N) + 1
burst = rng.random(N) < 0.02                              # velocity bursts
velocity = np.where(burst, rng.integers(13, 40, N), velocity)
print('base fields generated')

base fields generated


In [3]:
# --- failure simulation: logistic risk model ---
is_night = np.isin(hours, [0,1,2,3,4,5])
cat_effect = {'gaming':0.95,'travel':0.45,'electronics':0.20,'fashion':0.10,
              'food_delivery':0.0,'fuel':-0.20,'health':-0.25,'grocery':-0.50}
pay_effect = {'wallet':0.30,'net_banking':0.15,'credit_card':0.0,
              'debit_card':-0.05,'upi':-0.25}
logit = (-4.9
         + 0.30*np.log1p(amount)
         + 0.85*prior_fails
         + 0.80*is_high_risk
         + 0.55*is_night
         + 1.15*(auth_ms > 3000)
         + 0.75*(velocity > 12)
         + 0.70*(account_age < 30)
         + np.array([cat_effect[c] for c in merchant_cat])
         + np.array([pay_effect[p] for p in payment_method])
         + 0.35*(issuer_bank == 'BankF')
         + 0.25*((pd.to_datetime('2025-01-01') + pd.to_timedelta(days, unit='D')).dayofweek >= 5))
p_fail = 1 / (1 + np.exp(-logit))
is_failed = rng.random(N) < p_fail
print(f'simulated failure rate: {is_failed.mean():.2%}')
assert 0.02 < is_failed.mean() < 0.08, 'failure rate out of expected band'

reasons = ['insufficient_funds','issuer_decline','gateway_timeout',
           'suspected_fraud','invalid_credentials','duplicate_transaction']
failure_reason = np.where(
    is_failed,
    rng.choice(reasons, N, p=[0.32,0.24,0.18,0.14,0.07,0.05]),
    'none')
status = np.where(is_failed, 'failed', 'success')

simulated failure rate: 6.08%


In [4]:
# --- assemble, inject data-quality issues, save ---
df = pd.DataFrame({
    'transaction_id': 'TXN-' + pd.Series(np.arange(1_000_000, 1_000_000 + N)).astype(str),
    'timestamp': timestamps,
    'merchant_id': merchant_id,
    'merchant_category': merchant_cat,
    'payment_method': payment_method,
    'transaction_amount': np.round(amount, 2),
    'currency': currency,
    'country': country,
    'is_high_risk_country': is_high_risk,
    'device_type': device_type,
    'issuer_bank': issuer_bank,
    'authorization_response_time_ms': np.round(auth_ms).astype(int),
    'account_age_days': account_age,
    'prior_failed_attempts_24h': prior_fails,
    'txn_velocity_24h': velocity,
    'status': status,
    'failure_reason': failure_reason,
})
# realistic messiness: missing values + exact duplicates
for col, frac in [('device_type', 0.006), ('issuer_bank', 0.004), ('transaction_amount', 0.003)]:
    df.loc[rng.random(N) < frac, col] = np.nan
df = pd.concat([df, df.sample(600, random_state=7)], ignore_index=True)
df = df.sort_values('timestamp').reset_index(drop=True)
df.to_csv('../data/raw/transactions.csv', index=False)
print(f'saved ../data/raw/transactions.csv with {len(df):,} rows')

saved ../data/raw/transactions.csv with 180,600 rows


In [5]:
# --- inspection ---
raw = pd.read_csv('../data/raw/transactions.csv', parse_dates=['timestamp'])
print('shape:', raw.shape)
print('period:', raw['timestamp'].min(), '->', raw['timestamp'].max())
print('\nmissing values:')
print(raw.isna().sum()[raw.isna().sum() > 0])
print('\nduplicated rows:', raw.duplicated().sum())
print('\nstatus distribution:')
print(raw['status'].value_counts(normalize=True).round(4))
print('\nfailure reasons:')
print(raw.loc[raw['status']=='failed','failure_reason'].value_counts())
raw.head(3)

shape: (180600, 17)
period: 2025-01-01 00:03:00 -> 2025-12-31 23:48:00

missing values:
transaction_amount     496
device_type           1070
issuer_bank            724
dtype: int64



duplicated rows: 600

status distribution:
status
success    0.9392
failed     0.0608
Name: proportion, dtype: float64

failure reasons:
failure_reason
insufficient_funds       3600
issuer_decline           2586
gateway_timeout          1983
suspected_fraud          1494
invalid_credentials       766
duplicate_transaction     557
Name: count, dtype: int64


,transaction_id,timestamp,merchant_id,merchant_category,payment_method,transaction_amount,currency,country,is_high_risk_country,device_type,issuer_bank,authorization_response_time_ms,account_age_days,prior_failed_attempts_24h,txn_velocity_24h,status,failure_reason
0,TXN-1177473,2025-01-01 00:03:00,M0744,travel,debit_card,21.39,USD,IN,False,mobile,BankE,231,158,0,2,success,none
1,TXN-1015271,2025-01-01 00:09:00,M0542,electronics,wallet,15.26,USD,IN,False,mobile,BankE,690,298,2,3,success,none
2,TXN-1094413,2025-01-01 00:20:00,M0324,fuel,credit_card,62.04,USD,US,False,mobile,BankD,777,774,0,2,success,none


## What we have
- ~180k transactions across all of 2025, with timestamps, amounts, channels and behaviour signals.
- A small low-single-digit failure rate driven by interpretable risk factors.
- Realistic messiness on purpose: missing values and duplicate rows, which notebook 02 will explore and `src/data_cleaning.py` will fix.
- Next: **02_eda** - where do settlement failures concentrate?